# Module 2: Advanced Dependency Management

## Welcome & Module Induction
Welcome back! 

**What we learned previously:** In Module 1, we mastered the environment. We learned how to isolate our workspace using virtual environments and how to control the underlying operating system linkages (`$PATH` and `sys.path`). We effectively built a secure, isolated "room" for our code to execute without interference.

**What we are learning today:** Now, we focus on what we bring *into* that room. We are moving from simply running `pip install` to managing libraries like a professional DevOps engineer. We will cover strict version pinning, cryptographic lockfiles, and offline caching.

**Why this will be helpful:** In professional research and systems engineering—especially when benchmarking complex vision models like YOLO or running 3D segmentation architectures—a silent background update to a library like OpenCV or PyTorch can break your entire pipeline. Today, you will learn how to lock down your dependencies so your code is 100% reproducible on any machine or server, and how to cache massive libraries locally to save time in lab environments where internet speeds fluctuate.

Let's start by inspecting what is currently installed in our isolated environment.

In [5]:
import importlib.metadata
import sys

print("=== 1. Inspecting Installed Packages ===")

# Checking for the heavy data/AI libraries critical to vision and deep learning
critical_packages = ['numpy', 'torch', 'opencv-python', 'requests']

print(f"Python Version: {sys.version.split(' ')[0]}\n")

for pkg in critical_packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"✅ {pkg.ljust(15)} : Installed (v{version})")
    except importlib.metadata.PackageNotFoundError:
        print(f"❌ {pkg.ljust(15)} : NOT INSTALLED")

=== 1. Inspecting Installed Packages ===
Python Version: 3.11.3

✅ numpy           : Installed (v1.24.3)
❌ torch           : NOT INSTALLED
❌ opencv-python   : NOT INSTALLED
✅ requests        : Installed (v2.29.0)


## The Basic Standard: `requirements.txt`

A standard `pip freeze > requirements.txt` dumps every single package in your environment, including useless background tools. 

A professional approach is to identify exactly what your project needs and write it programmatically. More importantly, we use **Version Pinning**. Notice the difference between `requests` (which will download whatever the newest version is tomorrow) and `opencv-python==4.8.0.76` (which forces the system to download this exact version, preventing unexpected update errors).

In [6]:
import os

print("=== 2. Programmatic Requirements Generation ===")

# Define the exact dependencies needed for our pipeline
project_dependencies = [
    "numpy>=1.24.0,<2.0.0",  # Allow minor updates, but prevent major breaking changes
    "opencv-python==4.8.0.76", # Strictly pin vision libraries
    "requests"                 # Unpinned (not recommended for production)
]

req_file_path = "strict_requirements.txt"

with open(req_file_path, "w") as file:
    file.write("# Auto-generated strict dependencies\n")
    for dep in project_dependencies:
        file.write(f"{dep}\n")

print(f"Successfully generated {req_file_path}")

# Read it back to verify
with open(req_file_path, "r") as file:
    print("\nFile Contents:")
    print(file.read())

=== 2. Programmatic Requirements Generation ===
Successfully generated strict_requirements.txt

File Contents:
# Auto-generated strict dependencies
numpy>=1.24.0,<2.0.0
opencv-python==4.8.0.76
requests



## The Advanced Tier: Simulating Lockfile Integrity

Even if you pin a library, that library might secretly install a sub-dependency in the background. If that sub-dependency updates, your code still breaks. 

Modern tools like Poetry or Pipenv solve this using **Lockfiles** (`poetry.lock`). These tools calculate a massive tree of *every single sub-dependency* and lock them to an exact cryptographic hash. This guarantees the environment builds perfectly on any machine. 

Let's simulate how a system validates a package hash to ensure security and consistency.

In [7]:
import hashlib

print("=== 3. Package Integrity & Hash Verification ===")

def calculate_file_hash(filepath):
    """Simulates how Poetry or Pipenv calculates a lockfile hash."""
    sha256_hash = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            # Read and update hash string value in blocks of 4K
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except FileNotFoundError:
        return None

# We will hash the requirements file we just made as a demonstration
expected_hash = calculate_file_hash(req_file_path)

print(f"Target File: {req_file_path}")
print(f"Generated SHA-256 Hash:\n{expected_hash}")
print("\nIn production, this hash is stored in a .lock file. If the file changes even slightly, the hash changes, and the installation will be blocked to protect your environment.")

=== 3. Package Integrity & Hash Verification ===
Target File: strict_requirements.txt
Generated SHA-256 Hash:
4b43de84d1e95b05e9f6e07ac2b2b1fa7ba2501416b30304028e45f3051c9f32

In production, this hash is stored in a .lock file. If the file changes even slightly, the hash changes, and the installation will be blocked to protect your environment.


## Handling Slow Connections: Local Caching

When dealing with massive libraries, relying on the internet for every installation is risky and wastes bandwidth. 

We can configure Python's `pip` to prioritize a local offline cache folder. This allows a research team to download heavy libraries once, store them on a local drive, and install them rapidly across multiple machines without needing an active internet connection.

In [8]:
print("=== 4. Configuring Offline Caching ===")

# Define a local cache directory within the project
cache_dir = os.path.abspath("./local_pip_cache")
os.makedirs(cache_dir, exist_ok=True)

# Set the environment variable so pip knows to look here first
os.environ["PIP_CACHE_DIR"] = cache_dir

print(f"Pip Cache Directory explicitly set to:\n-> {cache_dir}\n")

print("Terminal Command to download without installing (for offline transfer):")
print(f"pip download -d {cache_dir} numpy torch\n")

print("Terminal Command to install strictly from the offline cache:")
print(f"pip install --no-index --find-links={cache_dir} numpy torch")

=== 4. Configuring Offline Caching ===
Pip Cache Directory explicitly set to:
-> d:\PSEB Intership\MindGigs\Phase 2\local_pip_cache

Terminal Command to download without installing (for offline transfer):
pip download -d d:\PSEB Intership\MindGigs\Phase 2\local_pip_cache numpy torch

Terminal Command to install strictly from the offline cache:
pip install --no-index --find-links=d:\PSEB Intership\MindGigs\Phase 2\local_pip_cache numpy torch
